In [ ]:
import os
import certifi
import requests
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import ChatPromptTemplate
from langchain import hub
from langchain.tools import tool


In [4]:
from langchain.agents import create_react_agent, AgentExecutor

In [44]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GOOGLE_API_KEY = os.getenv("GEMINI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY= os.getenv("WEATHERSTACK_API_KEY")

In [45]:
search_tool = TavilySearchResults(max_results=2)

In [46]:
def getWeather_data(city: str) -> str:
    
    """
    Get weather data for a given city.
    """
    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response= requests.get(url)

    data=response.json()

    if "current" not in data:
        return "City not found or weather data unavailable."
    
    return (
        f"Weather in {city}: \n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
        )

In [32]:
result= search_tool.invoke("Give me the latest news on AI")
result

[{'url': 'https://www.artificialintelligence-news.com',
  'content': 'Government & Public Sector AI\n\nJune 17, 2026\n\n# Google Cloud generative AI automates council planning operations\n\nArrow on the floor as AI investments by insurers are now expected to generate tangible business value beyond mere efficiency, requiring a change in strategy.\n\nWorld of Work\n\nJune 16, 2026\n\n# Insurers pivot AI strategy toward core risk underwriting\n\nResham Kotecha, Open Data Institute: How the EU can lead in AI\n\nAI and Us\n\nJune 16, 2026\n\n# EU publishes its AI content labelling playbook ahead of the AI Act’s August deadline\n\nAI in Action\n\nJune 15, 2026\n\n# HarmonyOS 7 steps into the AI gap Apple left open in China\n\nAccenture: Consumers show growing trust in AI shopping agents\n\nAI in Action\n\nJune 15, 2026\n\n# Accenture: Consumers show growing trust in AI shopping agents\n\nAI and Us [...] Retail & Logistics AI\n\nJune 19, 2026\n\n### Aviva deploys AI to stop £230M in sophistic

In [33]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    google_api_key=GOOGLE_API_KEY
)

In [34]:
response = llm.invoke("Day today")
response

AIMessage(content='Today is **Monday, June 10, 2024**.', response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-65304593-083b-4264-bfe2-651049fc4057-0')

In [35]:
#Prompt
prompt= hub.pull("hwchase17/react")

c:\Users\Ron\anaconda3\envs\langchainAgent\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [19]:
tools = [search_tool]

In [18]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

Create Agent

In [36]:
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt

)

Create Executor

In [37]:
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True) 

Execution

In [ ]:
response = agent_executor.invoke({
    "input": (
        "What is the Capital of Jharkhand?"
        "and the current weather")
    })



> Entering new AgentExecutor chain...
Action: tavily_search_results_json
Action Input: Capital of Jharkhand[{'url': 'https://byjus.com/social-science/capital-of-jharkhand', 'content': 'Ranchi is the capital of Jharkhand. Dumka is the sub-capital of Jharkhand. Ranchi was chosen as one of the cities to be developed as a smart city under the Government of India’s flagship program “Smart Cities Mission”. This article will explain some of the interesting facets of Jharkhand and its capital city Ranchi.\n\nCapital of Jharkhand\n\nCapital of Jharkhand\n\n## Ranchi – City of Waterfalls\n\n### Jharkhand – A Brief Overview\n\n### Jharkhand – Economy\n\n## Frequently Asked Questions (FAQs)\n\n### Is Jamshedpur the capital of Jharkhand?\n\n### Where is the second capital of Jharkhand?\n\n### What is the capital of Bihar?\n\n### What is the capital of Rajasthan?\n\n### What is the capital of Haryana?\n\n#### Comments\n\n### Leave a Comment Cancel reply'}, {'url': 'https://en.wikipedia.org/wiki/Jh

In [42]:
print(response["output"])

The capital of Jharkhand is Ranchi. The current weather in Ranchi as of 11:30 AM on June 22, 2026, is mist with a temperature of 24°C (feels like 25°C), 94% humidity, and a 68% chance of rain. The wind speed is 10.4 km/h from WNW, and the visibility is 2 km.
